# Comparison of Trefftz-Based PINNs and Standard PINNs Focusing on Structure Preservation

**Paper:** Koyamada, K., Ohtani, H. (2026). *Comparison of Trefftz-Based PINNs and Standard PINNs Focusing on Structure Preservation.* Journal of Advanced Simulation in Science and Engineering.

**Carpeta origen:** `PINNs/3. Arquitecturas, frameworks y variantes/Comparison of Trefftz-Based PINNs and Standard.pdf`

## Como se usan las PINNs en este paper

El paper documenta el fenomeno de **"alucinacion de residuo"** (*residual hallucination*): una PINN estandar puede converger a un residuo de EDP casi nulo y aun asi producir una solucion **fisicamente incorrecta**, porque el Teorema de Aproximacion Universal garantiza convergencia de la funcion pero no de sus derivadas de segundo orden. Su ejemplo motivador (Fig. 2) es exactamente la **ecuacion de Laplace 2D** de conduccion de calor estacionaria: $\nabla^2\Phi=0$. Comparan tres modelos: la solucion exacta, una red puramente supervisada (datos), y una PINN estandar que minimiza el residuo &mdash; la PINN reduce el residuo a un valor casi nulo pero su campo de temperatura **se desvia claramente** de la solucion exacta.

Como remedio, proponen **Trefftz-PINN** (Eq. 3): en vez de que toda la solucion resida en una red neuronal sin restricciones, se representa

$$u(\mathbf{x})=\sum_{i=1}^{N_b} c_i\,\phi_i(\mathbf{x}) + u_{NN}(\mathbf{x})$$

donde $\phi_i(\mathbf{x})$ son **funciones base de Trefftz** que satisfacen la EDP gobernante *exactamente y por construccion* (para Laplace 2D: polinomios armonicos, p.ej. $1,\,x,\,y,\,x^2-y^2,\,2xy,\dots$), $c_i$ son coeficientes escalares entrenables, y $u_{NN}$ es una red residual pequena que solo corrige efectos de frontera/discretizacion. Como la parte dominante de la solucion ya vive en un espacio fisicamente admisible, Trefftz-PINN **preserva la estructura global** (superficies magneticas, topologia de lineas de corriente) incluso cuando el MSE puntual es identico al de una PINN estandar &mdash; el punto central del paper: *el MSE no basta para garantizar consistencia fisica*.

Este cuaderno reproduce fielmente el **ejemplo motivador del paper** (Fig. 2, Laplace 2D) con una solucion armonica exacta conocida: primero se **induce deliberadamente** la alucinacion de residuo en una PINN estandar (desbalanceando los pesos de perdida PDE/frontera, replicando la "patologia de gradiente" via NTK que el paper cita de Wang et al.), y luego se construye una **Trefftz-PINN** (Eq. 3) con base de polinomios armonicos que evita el problema por construccion.

## Repositorio publico de referencia

El PDF no incluye un repositorio de codigo propio, ni se encontro uno especifico al buscar en GitHub (es un articulo breve de conferencia). Como referencia general del framework PINN base:

- **maziarraissi/PINNs** &mdash; https://github.com/maziarraissi/PINNs

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Problema de Laplace 2D (Fig. 2 del paper) con solucion armonica exacta

In [ ]:
def exact_T(x, y):
    """T(x,y) = exp(pi x) sin(pi y): armonica exacta (Laplaciano = 0)."""
    return torch.exp(np.pi * x) * torch.sin(np.pi * y)

N_col = 2000
xy_col = torch.rand(N_col, 2, device=device, requires_grad=True)

N_b = 100
edges = []
for fixed_val, axis in [(0.0, 0), (1.0, 0), (0.0, 1), (1.0, 1)]:
    t = torch.rand(N_b)
    pts = torch.zeros(N_b, 2)
    pts[:, axis] = fixed_val
    pts[:, 1 - axis] = t
    edges.append(pts)
xy_bnd = torch.cat(edges, dim=0).to(device)
T_bnd = exact_T(xy_bnd[:, 0:1], xy_bnd[:, 1:2]).detach()


def d_d(f, v, idx):
    g = torch.autograd.grad(f, v, grad_outputs=torch.ones_like(f),
                             create_graph=True, retain_graph=True)[0]
    return g[:, idx:idx + 1]


def laplacian(u, xy):
    u_x = d_d(u, xy, 0)
    u_y = d_d(u, xy, 1)
    u_xx = d_d(u_x, xy, 0)
    u_yy = d_d(u_y, xy, 1)
    return u_xx + u_yy

## 2. PINN estandar: se induce deliberadamente la 'alucinacion de residuo' (Seccion 1.2, Fig. 2)

Siguiendo el propio diagnostico del paper (patologia de gradiente via NTK, Wang et al.), se desbalancea el peso PDE/frontera para reproducir el fallo: la red reduce el residuo a casi cero mientras la solucion se desvia claramente de la exacta.

In [ ]:
class StandardPINN(nn.Module):
    def __init__(self, n_hidden=4, n_neurons=40):
        super().__init__()
        layers = [nn.Linear(2, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, xy):
        return self.net(xy)


model_pinn = StandardPINN().to(device)
opt = torch.optim.Adam(model_pinn.parameters(), lr=1e-3)

lam_pde, lam_bc = 1000.0, 0.05  # desbalance deliberado: el residuo domina, la frontera casi no pesa
history_pinn = []
for epoch in range(3000):
    opt.zero_grad()
    T_pred = model_pinn(xy_col)
    lap = laplacian(T_pred, xy_col)
    loss_pde = torch.mean(lap**2)
    loss_bc = torch.mean((model_pinn(xy_bnd) - T_bnd)**2)
    loss = lam_pde * loss_pde + lam_bc * loss_bc
    loss.backward()
    opt.step()
    history_pinn.append(loss_pde.item())
    if epoch % 500 == 0:
        print(f'[PINN estandar] epoch {epoch:5d} | residuo_PDE={loss_pde.item():.4e} | bc={loss_bc.item():.4e}')

## 3. Trefftz-PINN (Eq. 3): base de polinomios armonicos + red residual pequena

In [ ]:
def harmonic_basis(xy):
    """Polinomios armonicos 2D hasta orden 4 (partes real/imaginaria de (x+iy)^n): satisfacen
    exactamente el Laplaciano = 0 en todo el dominio, por construccion."""
    x, y = xy[:, 0:1], xy[:, 1:2]
    z = torch.complex(x, y)
    basis = [torch.ones_like(x)]
    for n in range(1, 5):
        zn = z**n
        basis.append(zn.real)
        basis.append(zn.imag)
    return torch.cat(basis, dim=1)  # (N, 9)


class TrefftzPINN(nn.Module):
    def __init__(self, n_basis=9, n_hidden=3, n_neurons=20):
        super().__init__()
        self.coeffs = nn.Parameter(torch.zeros(n_basis))
        layers = [nn.Linear(2, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, xy):
        trefftz_part = harmonic_basis(xy) @ self.coeffs
        return trefftz_part.unsqueeze(-1) + 0.1 * self.net(xy)


model_trefftz = TrefftzPINN().to(device)
opt_t = torch.optim.Adam(model_trefftz.parameters(), lr=1e-2)

history_trefftz = []
for epoch in range(3000):
    opt_t.zero_grad()
    T_pred = model_trefftz(xy_col)
    lap = laplacian(T_pred, xy_col)
    loss_pde = torch.mean(lap**2)
    loss_bc = torch.mean((model_trefftz(xy_bnd) - T_bnd)**2)
    loss = lam_pde * loss_pde + lam_bc * loss_bc
    loss.backward()
    opt_t.step()
    history_trefftz.append(loss_pde.item())
    if epoch % 500 == 0:
        print(f'[Trefftz-PINN] epoch {epoch:5d} | residuo_PDE={loss_pde.item():.4e} | bc={loss_bc.item():.4e}')

## 4. Resultados: mismo nivel de residuo, estructura muy distinta (cf. Fig. 2-3 del paper)

In [ ]:
n_side = 60
xs = np.linspace(0, 1, n_side)
ys = np.linspace(0, 1, n_side)
Xg, Yg = np.meshgrid(xs, ys)
xy_grid = torch.tensor(np.stack([Xg.ravel(), Yg.ravel()], axis=1), dtype=torch.float32, device=device)

with torch.no_grad():
    T_exact = exact_T(xy_grid[:, 0:1], xy_grid[:, 1:2]).cpu().numpy().reshape(Xg.shape)
    T_pinn = model_pinn(xy_grid).cpu().numpy().reshape(Xg.shape)
    T_trefftz = model_trefftz(xy_grid).cpu().numpy().reshape(Xg.shape)

err_pinn = 100 * np.linalg.norm(T_pinn - T_exact) / np.linalg.norm(T_exact)
err_trefftz = 100 * np.linalg.norm(T_trefftz - T_exact) / np.linalg.norm(T_exact)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, field, title in zip(axes, [T_exact, T_pinn, T_trefftz],
                             ['Exacta', f'PINN estandar (err={err_pinn:.1f}%)',
                              f'Trefftz-PINN (err={err_trefftz:.1f}%)']):
    im = ax.contourf(Xg, Yg, field, levels=30, cmap='inferno')
    ax.set_title(title)
    plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print(f'Residuo final PDE -- PINN estandar: {history_pinn[-1]:.4e} | Trefftz-PINN: {history_trefftz[-1]:.4e}')
print(f'Error relativo L2  -- PINN estandar: {err_pinn:.2f}% | Trefftz-PINN: {err_trefftz:.2f}%')

Se espera observar (como en el paper) que **ambos modelos alcanzan un residuo de EDP muy pequeno**, pero que la **PINN estandar puede desviarse notablemente de la solucion exacta** (alucinacion de residuo) mientras la **Trefftz-PINN, al construir su solucion sobre una base que satisface Laplace por construccion, preserva la estructura fisica correcta** independientemente del desbalance de pesos de perdida usado durante el entrenamiento.